In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 7

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 3 (tau=59)
[ Info: [sliding] iter 1000/1000000 elapsed=5.6s, rate=0.190, mean=[0.753, 0.00324, 1.157], std=[0.0380, 0.000220, 0.1565] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=10.3s, rate=0.180, mean=[0.725, 0.00334, 1.117], std=[0.0377, 0.000314, 0.1178] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=14.3s, rate=0.156, mean=[0.740, 0.00253, 1.089], std=[0.0392, 0.001118, 0.1032] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=18.3s, rate=0.140, mean=[0.757, 0.00201, 1.063], std=[0.0439, 0.001253, 0.0989] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=22.2s, rate=0.131, mean=[0.762, 0.00172, 1.043], std=[0.0410, 0.001239, 0.0950] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=26.2s, rate=0.121, mean=[0.770, 0.00151, 1.032], std=[0.0412, 0.001206, 0.0898] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=30.2s, rate=0.115, mean=[0.778, 0.00135, 1.021], std=[0.0418, 0.001169, 0.0868] [ADAPT]
[ Info: [sliding] iter 8000/